<a href="https://colab.research.google.com/github/Glaze0/Assignment/blob/main/Question-sol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Start@8,05 PM IST ||

In [ ]:
# Data : https://drive.google.com/file/d/1yo2EaHHuTgex_U6IuJhunZwzSF1r1A_D/view?usp=drive_link

In [ ]:
# If we’re using the free API instead of using all the rows from the CSV dataset, just keep the limit to 10.

In [ ]:
# Project 1: Support Ticket Intelligence & Automation

# Problem Statement
# A large company receives hundreds of customer-support tickets every day. These tickets contain unstructured descriptions of customer problems such as password issues, payment failures, delivery delays, refunds, account problems, and technical issues.
# Currently, support teams manually read each ticket, understand the customer's problem, determine its urgency, identify the category, summarize the issue, and decide what action should be taken.
# Your task is to build a GenAI-powered Support Ticket Intelligence System using the Gemini API.
# The system should read each support ticket and automatically convert the unstructured text into structured, actionable information.

# Input
# You are provided with a CSV file containing customer-support tickets.
# The file contains the following columns:
# ticket_id — Unique identifier for each ticket
# ticket_text — The customer's unstructured support request

# Example:
# ticket_id,ticket_text
# T001,"I cannot reset my password. The reset link says it has expired."
# T002,"My card was charged twice for the same order."
# T003,"The app crashes every time I try to upload a photo."

# Expected Output
# For every support ticket, your system should generate the following information:
# ticket_id
# category
# priority
# sentiment
# issue
# summary
# recommended_action

# Example
# Input:
# Ticket ID: T001
# "I cannot reset my password. The reset link says it has expired.
# I need access urgently because I have a meeting tomorrow."

# Expected output format:
# Ticket ID: T001
# Category: Password Reset
# Priority: High
# Sentiment: Frustrated
# Issue: Password reset link has expired
# Summary: Customer is unable to reset their password.
# Recommended Action: Assist the customer with resetting their password.

# Requirements
# Your system should:
# Read the support tickets from the provided CSV file.
# Use the Gemini API to analyze each ticket.
# Extract the required information from the unstructured ticket text.
# Use Pydantic to define and validate the expected output structure.
# Process multiple support tickets automatically.
# Save the final results in a structured format such as CSV or JSON.

# Goal
# Build an automated pipeline that converts:
# Unstructured Support Ticket
#             ↓
#        Gemini API
#             ↓
#    Structured Information
#             ↓
#       Business-Ready Data

# The final system should allow a support team to quickly understand, prioritize, and act on customer tickets without manually reading every ticket.

In [ ]:
# Final deliverable

# A CSV containing the automatically analyzed tickets:

# ticket_id | category | priority | sentiment | issue | summary | recommended_action

In [ ]:
# Solution : https://colab.research.google.com/drive/19ser8r0BKYtXyXzcDVKyIUnOcGqUotdQ?usp=sharing

In [11]:
from google import genai
from google.genai import types
import csv
import pandas as pd
from google.colab import userdata
import json


In [3]:
data = pd.read_csv('/content/support_tickets_50.csv')

data.head()

,ticket_id,ticket_text,known_category,known_priority,known_sentiment
0,T001,I cannot reset my password. The reset link say...,Password Reset,High,Frustrated
1,T002,My card was charged twice for the same order.,Payment,High,Angry
2,T003,How can I change the email address associated ...,Account,Low,Neutral
3,T004,The app crashes every time I try to upload a p...,Technical Issue,High,Frustrated
4,T005,My order was supposed to arrive yesterday but ...,Delivery,High,Frustrated


In [9]:
prompt = f'''
  Read the unstructured data from: {data.head().to_string(index=False)}

  Extract and generate the following information as result:
   - ticket_id
   - category
   - priority
   - sentiment
   - issue
   - summary
   - recommended_action
'''

api_key = userdata.get('api-k')

client = genai.Client(api_key = api_key)

response = client.models.generate_content(
    model = 'gemini-3.5-flash',
    contents = prompt,
    config = types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema = {
            'type':'object',
            'properties':{
                'ticket_id':{
                    'type':'string'
                    },
                'category':{
                    'type':'string'
                    },
                'priority':{
                    'type':'string',
                    'enum':['low','medium','high']
                },
                'sentiment':{
                    'type':'string',
                    'enum':['positive','negative','neutral']
                },
                'issue':{
                    'type':'string'
                },
                'summary':{
                    'type':'string'
                },
                'recommended_action':{
                    'type':'string'
                }
            },
            'required':['ticket_id','category','priority','sentiment','issue','summary','recommended_action']
        },
        system_instruction='''
        you are an intellignet support ticket system. Generate a json output

        Rules:
        - Donot generate information.
        - Use only the given information to extract information
        - Preserve the names

        '''
    )
)

print(response.text)

{"ticket_id":"T001","category":"Password Reset","priority":"high","sentiment":"negative","issue":"I cannot reset my password. The reset link says it has expired.","summary":"Password reset link expired","recommended_action":"Send a new password reset link to the user"}


In [15]:
j = json.loads(response.text)
print(j)

{'ticket_id': 'T001', 'category': 'Password Reset', 'priority': 'high', 'sentiment': 'negative', 'issue': 'I cannot reset my password. The reset link says it has expired.', 'summary': 'Password reset link expired', 'recommended_action': 'Send a new password reset link to the user'}


In [17]:
for k,v in j.items():
  print(f'{k}: {v}')

ticket_id: T001
category: Password Reset
priority: high
sentiment: negative
issue: I cannot reset my password. The reset link says it has expired.
summary: Password reset link expired
recommended_action: Send a new password reset link to the user
